# NB21 — Ar-APT under the three seeds

After NB19 and NB20, the external result is the last claim in the thesis that still rests on a single training run: the 7.25 and 9.50 point gaps on the untransformed Ar-APT control, and the 11 to 13 point gap on the polished subsets.

**No training happens here.** The six checkpoints already exist — seed 42 from NB10 and NB14, seeds 1 and 2 from NB19 — and this notebook only rescores the prepared Ar-APT subset with each of them. Budget about 45 minutes of GPU.

**Output.** `nb21_arapt_seed_scores.parquet`, one row per Ar-APT item per seed, and a printed per-seed table with the paired bootstrap. It answers whether the external effect survives retraining, as the evasion effect did and the false-alarm effect did not.

In [ ]:
# ============================ 1/6  configuration ============================
import os, time, numpy as np, pandas as pd, torch

# ---- Ar-APT, already prepared by NB16 (add the NB16 notebook output as input) ----
P_ARAPT    = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_prepared.parquet"
P_ARAPT_V  = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_vstat.parquet"
P_ARAPT_CH = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/arapt_chunks_K9.parquet"

# ---- the six checkpoints: seed 42 from the production runs, seeds 1 and 2 from NB19 ----
CKPTS = {
    (42, "hybrid"): "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/ckpt/last.pt",
    (42, "neural"): "/kaggle/input/notebooks/bahaaqassem/nb14-neural-only-production/ckpt_neural/last.pt",
    (1,  "hybrid"): "/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication/nb19_hybrid_seed1.pt",
    (1,  "neural"): "/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication/nb19_neural_seed1.pt",
    (2,  "hybrid"): "/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication-seed2/nb19_hybrid_seed2.pt",
    (2,  "neural"): "/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication-seed2/nb19_neural_seed2.pt",
}

MODEL_ID   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS  = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]
HIDDEN, DROPOUT = 256, 0.0
N_BOOT = 2000
RNG = np.random.default_rng(42)
OUT = "/kaggle/working/nb21_arapt_seed_scores.parquet"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[1/6] device {DEV} | checkpoints declared {len(CKPTS)}", flush=True)
missing = [k for k, v in CKPTS.items() if not os.path.exists(v)]
for k, v in CKPTS.items():
    print(f"[1/6]   seed {k[0]:2d} {k[1]:6s} {'found  ' if os.path.exists(v) else 'MISSING'} {v}", flush=True)
assert not missing, f"add these notebook outputs as inputs first: {missing}"

In [ ]:
# ============================ 2/6  the prepared Ar-APT subset ============================
AR  = pd.read_parquet(P_ARAPT)
ARV = pd.read_parquet(P_ARAPT_V)
ARC = pd.read_parquet(P_ARAPT_CH)
AR  = AR.set_index("arapt_id").loc[ARV.arapt_id].reset_index()
ARC = ARC.set_index("arapt_id").loc[AR.arapt_id].reset_index()
Xa  = ARV[STAT_COLS].to_numpy(np.float32)
shape = tuple(ARC.chunk_shape.iloc[0])
CHa = np.array(ARC.chunks.tolist(), dtype=np.int32).reshape(len(ARC), *shape)
NCa = ARC.n_chunks.to_numpy()
print(f"[2/6] Ar-APT items {len(AR)} | chunks {CHa.shape} | features {Xa.shape}", flush=True)
print(f"[2/6] kinds: {AR.kind.value_counts().to_dict()}", flush=True)
if "level" in AR.columns:
    print(f"[2/6] polishing levels: {AR[AR.kind=='human_polished'].level.value_counts().sort_index().to_dict()}", flush=True)
assert len(AR) == len(CHa) == len(Xa), "the three Ar-APT files are not aligned"

In [ ]:
# ============================ 3/6  the model, as defined in NB17 ============================
!pip install -q transformers
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
tok_c = AutoTokenizer.from_pretrained(MODEL_ID); PAD = tok_c.pad_token_id

class Net(nn.Module):
    def __init__(self, use_stat, stat_dim=5):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        h = self.enc.config.hidden_size; fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        B, K, L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:, 0, :].view(B, K, -1)
        m = (torch.arange(K, device=ids.device)[None, :] < nch[:, None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        if self.use_stat: v = torch.cat([v, stat], 1)
        v = (v - self.mu)/self.sd
        return self.head(v)

@torch.no_grad()
def predict(model, CHm, NCm, Xm, tag):
    model.eval(); P = []; t0 = time.time()
    for i in range(0, len(CHm), 16):
        ids = torch.from_numpy(CHm[i:i+16].astype(np.int64)).to(DEV)
        nch = torch.tensor(NCm[i:i+16].astype(np.int64)).to(DEV)
        st  = torch.from_numpy(Xm[i:i+16]).to(DEV)
        with torch.amp.autocast('cuda'):
            p = torch.softmax(model(ids, nch, st), 1)[:, 1]
        P.append(p.float().cpu().numpy())
        if (i//16) % 40 == 0 and i:
            print(f"[3/6]   {tag} {i}/{len(CHm)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(P)
print("[3/6] model and prediction routine ready", flush=True)

In [ ]:
# ============================ 4/6  score with every checkpoint ============================
frames = []
for (seed, name), ck in CKPTS.items():
    sd = torch.load(ck, map_location=DEV, weights_only=False)
    state = sd["model"] if isinstance(sd, dict) and "model" in sd else sd
    m = Net(name == "hybrid").to(DEV)
    m.load_state_dict(state)
    fused = m.mu.numel()
    print(f"[4/6] seed {seed} {name}: loaded, fused dimension {fused}", flush=True)
    assert fused == (773 if name == "hybrid" else 768), "checkpoint does not match the declared model type"
    p = predict(m, CHa, NCa, Xa, f"seed{seed}/{name}")
    frames.append(pd.DataFrame({"arapt_id": AR.arapt_id.values, "seed": seed, "model": name, "p": p}))
    print(f"[4/6] seed {seed} {name}: scored {len(p)} items", flush=True)
    del m; torch.cuda.empty_cache()

S = pd.concat(frames, ignore_index=True)
W = S.pivot_table(index=["arapt_id", "seed"], columns="model", values="p").reset_index()
W = W.rename(columns={"hybrid": "p_hybrid", "neural": "p_neural"})
meta = AR[[c for c in ("arapt_id", "kind", "level", "model", "domain") if c in AR.columns]]
W = W.merge(meta, on="arapt_id", how="left", suffixes=("", "_meta"))
W.to_parquet(OUT, index=False)
print(f"[4/6] wrote {OUT} | rows {len(W)} | seeds {sorted(W.seed.unique())}", flush=True)

In [ ]:
# ============================ 5/6  rates per seed ============================
def rate(p): return 100.0*float((np.asarray(p) >= 0.5).mean()) if len(p) else float("nan")

rows = []
for seed, s in W.groupby("seed"):
    ho = s[s.kind == "human_orig"]; ai = s[s.kind == "ai_generated"]; hp = s[s.kind == "human_polished"]
    rows.append({"seed": int(seed),
                 "n_human": len(ho), "FPR_hyb": round(rate(ho.p_hybrid), 2), "FPR_neu": round(rate(ho.p_neural), 2),
                 "n_ai": len(ai),    "TPR_hyb": round(rate(ai.p_hybrid), 2), "TPR_neu": round(rate(ai.p_neural), 2),
                 "n_pol": len(hp),   "POL_hyb": round(rate(hp.p_hybrid), 2), "POL_neu": round(rate(hp.p_neural), 2)})
R = pd.DataFrame(rows)
R["gap_FPR"] = (R.FPR_hyb - R.FPR_neu).round(2)
R["gap_TPR"] = (R.TPR_hyb - R.TPR_neu).round(2)
R["gap_POL"] = (R.POL_hyb - R.POL_neu).round(2)
print("[5/6] Ar-APT rates per seed, hybrid against neural", flush=True)
print(R.to_string(index=False), flush=True)
print("", flush=True)
print("[5/6] thesis reports, from seed 42 alone: control FPR gap -7.25, control TPR gap -9.50", flush=True)
for col, label in (("gap_FPR", "untransformed human (false alarms)"),
                   ("gap_TPR", "untransformed machine (detection)"),
                   ("gap_POL", "polished human (false alarms)")):
    v = R[col]
    print(f"[5/6] {label:36s} mean {v.mean():+6.2f}  sd {v.std(ddof=1):5.2f}  "
          f"range [{v.min():+6.2f}, {v.max():+6.2f}]  sign held {bool((v<0).all() or (v>0).all())}", flush=True)

if "level" in W.columns:
    print("\n[5/6] polished subsets by level and seed (hybrid minus neural)", flush=True)
    for (seed, lv), s in W[W.kind == "human_polished"].groupby(["seed", "level"]):
        print(f"[5/6]   seed {int(seed):2d} level {lv}: n={len(s):4d} "
              f"hybrid {rate(s.p_hybrid):6.2f}  neural {rate(s.p_neural):6.2f}  "
              f"gap {rate(s.p_hybrid)-rate(s.p_neural):+6.2f}", flush=True)

In [ ]:
# ============================ 6/6  paired bootstrap and verdict ============================
def boot(sub, label):
    h = (sub.p_hybrid.to_numpy() >= 0.5).astype(float)
    n = (sub.p_neural.to_numpy() >= 0.5).astype(float)
    if len(h) == 0: return None
    obs = 100.0*(h.mean() - n.mean())
    idx = RNG.integers(0, len(h), size=(N_BOOT, len(h)))
    d = 100.0*(h[idx].mean(1) - n[idx].mean(1))
    lo, hi = np.percentile(d, [2.5, 97.5])
    sig = "yes" if (lo > 0) or (hi < 0) else "no"
    print(f"[6/6] {label:40s} gap {obs:+6.2f}  95% CI [{lo:+6.2f}, {hi:+6.2f}]  significant {sig}", flush=True)
    return {"scope": label, "n": len(h), "gap": round(obs, 2),
            "lo": round(float(lo), 2), "hi": round(float(hi), 2), "significant": sig}

res = []
for seed, s in W.groupby("seed"):
    res.append(boot(s[s.kind == "human_orig"],     f"seed {int(seed)} · untransformed human"))
    res.append(boot(s[s.kind == "ai_generated"],   f"seed {int(seed)} · untransformed machine"))
    res.append(boot(s[s.kind == "human_polished"], f"seed {int(seed)} · polished human"))
print("", flush=True)
res.append(boot(W[W.kind == "human_orig"],     "all seeds pooled · untransformed human"))
res.append(boot(W[W.kind == "ai_generated"],   "all seeds pooled · untransformed machine"))
res.append(boot(W[W.kind == "human_polished"], "all seeds pooled · polished human"))
B = pd.DataFrame([r for r in res if r])

R.to_parquet("/kaggle/working/nb21_arapt_per_seed.parquet", index=False)
B.to_parquet("/kaggle/working/nb21_arapt_bootstrap.parquet", index=False)
print("\n[6/6] wrote nb21_arapt_per_seed.parquet and nb21_arapt_bootstrap.parquet", flush=True)

per = [r for r in res if r and r["scope"].startswith("seed")]
print("[6/6] READ THIS AS FOLLOWS", flush=True)
print(f"[6/6]   per-seed comparisons significant: {sum(1 for r in per if r['significant']=='yes')} of {len(per)}", flush=True)
print("[6/6]   if a sign holds across all three seeds, the external claim can stand as written;", flush=True)
print("[6/6]   if it flips on any seed, report it as a range across seeds, as the stress test now does.", flush=True)
print(B.to_string(index=False), flush=True)